# Realtime Telemetry, Joints and Services

Read what the robot is doing right now: polled joint servo positions (T:106 at 1 Hz inside the main app), battery, host sensors, the host error ring, and the state of the onboard voice/VLM services.

The same data powers the web UI **Telemetry** tab.

In [ ]:
import requests, time

BASE = "http://127.0.0.1:5000"
LABELS = ["LF_A", "LF_B", "LF_H", "LH_A", "LH_B", "LH_H",
          "RF_A", "RF_B", "RF_H", "RH_A", "RH_B", "RH_H"]

for i in range(5):
    d = requests.get(BASE + "/api/telemetry", timeout=3).json()
    fb = (d.get("joints") or {}).get("fb") or []
    print(f"--- frame {i+1}  battery={d.get('battery')}V  cpu={d.rpi.cpu_load}% ---")
    for name, v in zip(LABELS, fb):
        bar = "#" * (v * 30 // 1023)
        print(f"{name:5s} {v:4d} {bar}")
    time.sleep(1.0)

In [ ]:
d = requests.get(BASE + "/api/telemetry", timeout=3).json()
errs = d.get("errors", [])
print("host errors:", len(errs))
for e in errs[:10]:
    print(f"[{e['level']}] {e['source']}: {e['msg'][:120]}")

## Onboard services

The Pi runs three services: the main app (cron @reboot), the voice assistant and the VLM goal service (systemd).

In [ ]:
import subprocess
for svc in ("wavego-voice", "wavego-vlm"):
    r = subprocess.run(["systemctl", "is-active", svc], capture_output=True, text=True)
    print(f"{svc}: {r.stdout.strip()}")
print("main app: cron @reboot -> check ~/ugv.log")